# ECCE 2027 Full Run -- TRAIN ONLY (Kaggle): E2.2 -- Chattogram Fold 2 (train) -> out-of-fold + reverse-transfer eval

Split into train-only / eval-only notebooks: inference on this Kaggle T4 x2
tier leaks GPU memory when run in the same process as training, so evaluation
always happens in a separate, fresh notebook (`eval_run_e2_2.ipynb`).

**Attach one dataset before running** (Add Input, right sidebar):
- `badodd-ecce2027-bundle` (or whatever you named it) -- contains `badodd.zip`
  (images) and `badodd_ecce2027_overlay.zip` (labels/splits/configs).

Settings: Accelerator = GPU T4 x2, Internet = ON.

Train manifest: `splits/ctg_fold2_train.txt`
Monitor manifest: `splits/ctg_fold2_train_monitor.txt` (10% slice of the training set, val-only, never a real test set)

**When this finishes:** click **Save Version -> Save & Run All (Commit)**, then
attach this notebook as an input to `eval_run_e2_2.ipynb`.


In [ ]:
# ==============================================================================
# 1. HARDWARE & ENVIRONMENT VERIFICATION
# ==============================================================================
import os, sys, time, glob, json, shutil, subprocess

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from pathlib import Path
import torch

PINNED_ULTRALYTICS = "8.4.155"
RUN_ID = 'e2_2'

print('Python version:', sys.version)
print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
print('CUDA device count (should be 1 after masking):', torch.cuda.device_count())

subprocess.run("nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free "
               "--format=csv", shell=True)

assert torch.cuda.is_available(), (
    'FAIL: no GPU detected -- Accelerator is set to "None" in this notebook\'s Settings tab. '
    'Training on CPU would take ~27 hours for 100 epochs (vs ~40 min on a T4) and cannot finish '
    'inside Kaggle\'s 12-hour session limit. Cancel this run, set Accelerator = GPU T4 x2 in '
    'Settings, and re-run Save & Run All.'
)
device_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU Device: {device_name} ({total_vram:.2f} GB VRAM)')
torch.cuda.reset_peak_memory_stats(0)

subprocess.run(f"pip install -q ultralytics=={PINNED_ULTRALYTICS} pycocotools", shell=True, check=True)
import ultralytics
assert ultralytics.__version__ == PINNED_ULTRALYTICS, (
    f"Ultralytics version drift: installed {ultralytics.__version__}, expected {PINNED_ULTRALYTICS}"
)
print('Ultralytics version:', ultralytics.__version__)
print('Run ID:', RUN_ID)

from ultralytics import YOLO

In [ ]:
# ==============================================================================
# 2. DATASET & OVERLAY DISCOVERY -- FAIL HARD on any missing image
# ==============================================================================
print('=== Scanning /kaggle/input for Dataset & Overlay ===')

badodd_zips = glob.glob('/kaggle/input/**/badodd.zip', recursive=True)
if badodd_zips and not glob.glob('/kaggle/input/**/*.jpg', recursive=True):
    print(f'Found badodd.zip at {badodd_zips[0]}. Extracting to /kaggle/working/badodd_images...')
    os.makedirs('/kaggle/working/badodd_images', exist_ok=True)
    subprocess.run(f"unzip -q {badodd_zips[0]} -d /kaggle/working/badodd_images", shell=True, check=True)
    IMAGE_SEARCH_ROOT = '/kaggle/working/badodd_images'
else:
    IMAGE_SEARCH_ROOT = '/kaggle/input'

all_input_imgs = [p for p in glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.jpg', recursive=True) +
                        glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.png', recursive=True)
                   if not os.path.basename(p).startswith('._')]
img_lookup = {os.path.basename(p): p for p in all_input_imgs}
print(f'Indexed {len(img_lookup)} images from {IMAGE_SEARCH_ROOT}.')
assert len(img_lookup) >= 10000, (
    f"Expected ~10,032 BadODD images, found only {len(img_lookup)} -- "
    f"is the bundle dataset attached?"
)

overlay_roots = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'splits' in dirs and 'labels' in dirs:
        overlay_roots.append(root)

overlay_zips = glob.glob('/kaggle/input/**/badodd_ecce2027_overlay*.zip', recursive=True)
overlay_src = None
if overlay_roots:
    overlay_src = overlay_roots[0]
    print(f'Found overlay directory at: {overlay_src}')
elif overlay_zips:
    print(f'Found overlay zip at: {overlay_zips[0]}. Unzipping to /kaggle/working/overlay...')
    subprocess.run(f"unzip -q {overlay_zips[0]} -d /kaggle/working/overlay", shell=True, check=True)
    overlay_src = '/kaggle/working/overlay'
else:
    raise FileNotFoundError(
        'Could not locate the ECCE 2027 overlay package in /kaggle/input! '
        'Attach the bundle dataset.'
    )

assert os.path.isdir(os.path.join(overlay_src, 'labels'))
assert os.path.isdir(os.path.join(overlay_src, 'splits'))
assert os.path.isdir(os.path.join(overlay_src, 'configs'))

WORK_DATA = '/kaggle/working/data'
WORK_SPLITS = os.path.join(WORK_DATA, 'splits')
WORK_LABELS = os.path.join(WORK_DATA, 'labels')
WORK_IMAGES = os.path.join(WORK_DATA, 'images')
os.makedirs(WORK_SPLITS, exist_ok=True)
os.makedirs(WORK_LABELS, exist_ok=True)
os.makedirs(WORK_IMAGES, exist_ok=True)

label_files = glob.glob(os.path.join(overlay_src, 'labels', '*.txt'))
for lf in label_files:
    dest = os.path.join(WORK_LABELS, os.path.basename(lf))
    if not os.path.exists(dest):
        try:
            os.symlink(lf, dest)
        except OSError:
            shutil.copy(lf, dest)
print(f'Linked {len(label_files)} label files.')

for bname, src_p in img_lookup.items():
    dest = os.path.join(WORK_IMAGES, bname)
    if not os.path.exists(dest):
        try:
            os.symlink(src_p, dest)
        except OSError:
            shutil.copy(src_p, dest)
print(f'Linked {len(img_lookup)} image files.')

manifest_files = glob.glob(os.path.join(overlay_src, 'splits', '*.txt'))
resolved_counts = {}
for mf in manifest_files:
    mname = os.path.basename(mf)
    with open(mf, 'r') as f:
        bases = [os.path.basename(l.strip()) for l in f if l.strip()]
    resolved = []
    missing_here = []
    for b in bases:
        p = os.path.join(WORK_IMAGES, b)
        if os.path.exists(p):
            resolved.append(p)
        else:
            missing_here.append(b)
    if missing_here:
        raise FileNotFoundError(
            f"{mname}: {len(missing_here)} images referenced in the manifest are missing, "
            f"e.g. {missing_here[:5]}. Do not proceed -- check the attached dataset."
        )
    with open(os.path.join(WORK_SPLITS, mname), 'w') as f:
        f.writelines(p + '\n' for p in resolved)
    resolved_counts[mname] = len(resolved)

print(f'Resolved {len(manifest_files)} manifests, ALL images present. This run uses:')
for k in ('ctg_fold2_train.txt', 'ctg_fold2_train_monitor.txt'):
    print(f'  {k}: {resolved_counts.get(k)}')

print('Dataset & Overlay setup complete!')

In [ ]:
# ==============================================================================
# 3. WRITE YOLO DATASET CONFIG & LOAD PINNED HYPERPARAMETERS
# ==============================================================================
import yaml

with open(os.path.join(overlay_src, 'configs', 'hyp.yaml')) as f:
    hyp = yaml.safe_load(f)
assert hyp['ultralytics_version'] == PINNED_ULTRALYTICS, (
    f"hyp.yaml pins ultralytics=={hyp['ultralytics_version']} but this runtime has {PINNED_ULTRALYTICS}"
)
print('Loaded pinned hyperparameters from hyp.yaml:')
print(hyp)

CLASS_NAMES = {
    0: 'auto_rickshaw', 1: 'bicycle', 2: 'bus', 3: 'car', 4: 'cart_vehicle',
    5: 'construction_vehicle', 6: 'motorbike', 7: 'person', 8: 'priority_vehicle',
    9: 'three_wheeler', 10: 'truck',
}

run_yaml = {
    'path': WORK_DATA,
    'train': os.path.join(WORK_SPLITS, 'ctg_fold2_train.txt'),
    'val': os.path.join(WORK_SPLITS, 'ctg_fold2_train_monitor.txt'),
    'names': CLASS_NAMES,
}

RUN_YAML_PATH = f'/kaggle/working/data_{RUN_ID}.yaml'
with open(RUN_YAML_PATH, 'w') as f:
    yaml.dump(run_yaml, f, sort_keys=False)

print(f'\nWrote dataset configuration to: {RUN_YAML_PATH}')
print('Train manifest:', run_yaml['train'])
print('Val monitoring manifest:', run_yaml['val'])

In [ ]:
# ==============================================================================
# 3b. LABEL INTEGRITY AUDIT -- run BEFORE training. Catches silent background-only runs.
# ==============================================================================
train_manifest = run_yaml['train']
with open(train_manifest) as f:
    train_imgs = [l.strip() for l in f if l.strip()]

total_boxes_gt = 0
n_background = 0
for img_p in train_imgs:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(WORK_LABELS, stem + '.txt')
    if not os.path.exists(lbl_p) or os.path.getsize(lbl_p) == 0:
        n_background += 1
        continue
    with open(lbl_p) as f:
        n = sum(1 for line in f if line.strip())
    total_boxes_gt += n
    if n == 0:
        n_background += 1

bg_rate = n_background / len(train_imgs)
print(f'Train Images in Manifest: {len(train_imgs)} | Total Ground Truth Boxes: {total_boxes_gt} | '
      f'Background Images: {n_background} ({bg_rate*100:.2f}%)')
assert total_boxes_gt > 0, 'FAIL: zero ground-truth boxes found -- labels did not link correctly.'
assert bg_rate < 0.05, f'FAIL: background image rate {bg_rate*100:.2f}% >= 5% -- label linkage is broken.'
print('Label integrity check PASSED.')

In [ ]:
# ==============================================================================
# 4. RUN FULL TRAINING (E2.2 -- Chattogram Fold 2 (train) -> out-of-fold + reverse-transfer eval -- 100 EPOCHS) & SAVE EVERYTHING TO OUTPUT
# ==============================================================================
PROJECT_DIR = '/kaggle/working/runs/train'
EXP_NAME = RUN_ID
EXP_DIR = os.path.join(PROJECT_DIR, EXP_NAME)
LAST_CKPT = os.path.join(EXP_DIR, 'weights', 'last.pt')

can_resume = os.path.exists(LAST_CKPT)
model = YOLO(LAST_CKPT if can_resume else hyp['model'])
print('Resuming from checkpoint.' if can_resume else f"Starting fresh from {hyp['model']}.")
if can_resume:
    print('NOTE: resumed from a prior partial run -- total_train_time_s below only covers this session.')

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)

EPOCHS = 100
t0_train = time.time()
train_results = model.train(
    data=RUN_YAML_PATH,
    epochs=EPOCHS,
    patience=hyp['patience'],
    batch=hyp['batch'],
    imgsz=hyp['imgsz'],
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    optimizer=hyp['optimizer'],
    lr0=hyp['lr0'], lrf=hyp['lrf'], momentum=hyp['momentum'], weight_decay=hyp['weight_decay'],
    warmup_epochs=hyp['warmup_epochs'], warmup_momentum=hyp['warmup_momentum'], warmup_bias_lr=hyp['warmup_bias_lr'],
    box=hyp['box'], cls=hyp['cls'], dfl=hyp['dfl'],
    hsv_h=hyp['hsv_h'], hsv_s=hyp['hsv_s'], hsv_v=hyp['hsv_v'],
    degrees=hyp['degrees'], translate=hyp['translate'], scale=hyp['scale'], shear=hyp['shear'],
    perspective=hyp['perspective'], flipud=hyp['flipud'], fliplr=hyp['fliplr'],
    mosaic=hyp['mosaic'], mixup=hyp['mixup'], copy_paste=hyp['copy_paste'],
    seed=hyp['seed'], deterministic=hyp['deterministic'],
    project=PROJECT_DIR, name=EXP_NAME, exist_ok=True, save=True, resume=can_resume, verbose=True,
)

total_train_time = time.time() - t0_train
time_per_epoch = total_train_time / EPOCHS

peak_vram_gb = 0.0
if torch.cuda.is_available():
    peak_vram_gb = torch.cuda.max_memory_allocated(0) / (1024**3)

assert os.path.exists(LAST_CKPT), f'ERROR: {LAST_CKPT} was not created!'
print('\n' + '='*60)
print(f'           {RUN_ID} TRAINING RESULTS ({EPOCHS} epochs)           ')
print('='*60)
print(f'Total Training Time: {total_train_time:.2f} s ({total_train_time/60:.1f} min)')
print(f'Approx Duration Per Epoch (this session): {time_per_epoch:.2f} s')
print(f'Resumed from a prior partial run: {can_resume}')
print(f'Peak GPU VRAM Allocated: {peak_vram_gb:.2f} GB')
print(f'Checkpoint Generated: {LAST_CKPT}')
print('='*60)

run_info = {
    'run_id': RUN_ID,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'total_vram_gb': torch.cuda.get_device_properties(0).total_memory/(1024**3) if torch.cuda.is_available() else None,
    'torch_version': torch.__version__,
    'ultralytics_version': PINNED_ULTRALYTICS,
    'epochs': EPOCHS,
    'resumed': can_resume,
    'total_train_time_s': total_train_time,
    'seconds_per_epoch_this_session': time_per_epoch,
    'peak_vram_gb': peak_vram_gb,
}

# Save everything the matching eval notebook will need, under stable,
# unambiguous filenames so it can glob for them among whatever else is attached.
RESULTS_DIR = '/kaggle/working/results'
os.makedirs(f'{RESULTS_DIR}/logs', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/weights', exist_ok=True)
with open(f'{RESULTS_DIR}/logs/{RUN_ID}_run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)
shutil.copy(os.path.join(EXP_DIR, 'results.csv'), f'{RESULTS_DIR}/logs/{RUN_ID}_results.csv')
shutil.copy(os.path.join(EXP_DIR, 'args.yaml'), f'{RESULTS_DIR}/logs/{RUN_ID}_args.yaml')
shutil.copy(LAST_CKPT, f'{RESULTS_DIR}/weights/{RUN_ID}_last.pt')

print(json.dumps(run_info, indent=2))
print(f"\nSaved checkpoint to {RESULTS_DIR}/weights/{RUN_ID}_last.pt -- this is what")
print(f"eval_run_{RUN_ID}.ipynb will load. Click Save Version -> Save & Run All (Commit) now.")